ARTI308 - Machine Learning

# Credit Card Customer Segmentation Project

In this project, you will use K-Means clustering to segment [credit card customers](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata/data) based on their usage behavior. This is an unsupervised learning problem because the dataset does not contain a target label for customer groups.

You will use the `CC_GENERAL.csv` dataset.

## About the Dataset

The dataset contains customer-level credit card usage behavior. Each row represents one credit card holder, and the columns describe different behavioral variables such as balance, purchases, cash advance, payments, and tenure. The goal is to group similar customers together so that the company can understand different customer segments and design better marketing strategies.

## Import Libraries

**Import the libraries you need for data analysis, visualization, preprocessing, clustering, and evaluation.**

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

## Get the Data

**Read the `CC_GENERAL.csv` file and save it in a dataframe called `df`.**

In [2]:
 df = pd.read_csv("CC GENERAL.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'CC GENERAL.csv'

**Check the first five rows of the dataset.**

In [ ]:
df.head()

**Check the shape of the dataset.**

In [ ]:
df.shape

**Check basic information about the dataset using `info()`.**

In [ ]:
df.info()

**Check summary statistics using `describe()`.**

In [ ]:
df.describe().T

## Data Cleaning

The column `CUST_ID` is an identification column. It is not useful for clustering because it does not describe customer behavior.

**Drop the `CUST_ID` column from the dataframe.**

In [ ]:
customer_ids = df["CUST_ID"].copy()

df = df.drop(columns=["CUST_ID"])

df.head()

**Check the missing values in each column.**

In [ ]:
df.isnull().sum().sort_values(ascending=False)

Some columns may contain missing values.

Hint: You can handle missing values by either:
- filling them with the mean value
- or dropping the rows that contain missing values

For this project, use mean imputation.

**Fill the missing values with the mean of each column.**

In [ ]:
df = df.fillna(df.mean(numeric_only=True))

**Check the missing values again to make sure they were handled.**

In [ ]:
df.isnull().sum().sort_values(ascending=False)

## Exploratory Data Analysis

Before applying clustering, it is important to understand the data.

**Create histograms for the numerical columns.**

In [ ]:
df.hist(figsize=(18, 14), bins=30)
plt.suptitle("Histograms of Credit Card Customer Features", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

**Create a correlation heatmap to understand relationships between the features.**

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", square=False)
plt.title("Correlation Heatmap of Customer Behavior Features")
plt.tight_layout()
plt.show()

**Create a scatter plot between `BALANCE` and `PURCHASES`.**

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="BALANCE", y="PURCHASES", alpha=0.6)
plt.title("BALANCE vs PURCHASES")
plt.xlabel("Balance")
plt.ylabel("Purchases")
plt.tight_layout()
plt.show()

**Create a scatter plot between `BALANCE` and `CASH_ADVANCE`.**

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="BALANCE", y="CASH_ADVANCE", alpha=0.6)
plt.title("BALANCE vs CASH_ADVANCE")
plt.xlabel("Balance")
plt.ylabel("Cash Advance")
plt.tight_layout()
plt.show()

## Feature Scaling

K-Means is a distance-based algorithm. Therefore, feature scaling is very important.

The features in this dataset have very different ranges. For example, `BALANCE`, `PURCHASES`, and `CREDIT_LIMIT` may have large values, while frequency columns are between 0 and 1.

**Use StandardScaler to scale the data. Save the scaled data in a variable called `X_scaled`.**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

X_scaled_df = pd.DataFrame(X_scaled, columns=df.columns)
X_scaled_df.head()

## Choosing K Intuitively

Choosing K is one of the most difficult parts of K-Means.

Since this dataset has many features, it is not easy to visually see the clusters directly.

However, we can still compare different K values using the elbow method and silhouette score.

## Elbow Method

**Create a loop that fits K-Means models for K values from 1 to 10. Save the inertia values in a list called `inertia_values`.**

In [ ]:
inertia_values = []
k_values = range(1, 11)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

inertia_values

**Plot the elbow curve.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertia_values, marker="o")
plt.title("Elbow Method for Choosing K")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.xticks(list(k_values))
plt.grid(True)
plt.tight_layout()
plt.show()

**Output Interpretation**

Look at the elbow curve and try to identify where the decrease in inertia starts to slow down.

That point can suggest a reasonable value for K.

## Silhouette Score

The silhouette score helps evaluate how well-separated the clusters are.

**Create a loop that calculates the silhouette score for K values from 2 to 10. Save the scores in a list called `silhouette_scores`.**

In [ ]:
silhouette_scores = []
silhouette_k_values = range(2, 11)

for k in silhouette_k_values:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    cluster_labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, cluster_labels)
    silhouette_scores.append(score)

silhouette_scores

**Plot the silhouette scores.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(list(silhouette_k_values), silhouette_scores, marker="o")
plt.title("Silhouette Scores for Different K Values")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.xticks(list(silhouette_k_values))
plt.grid(True)
plt.tight_layout()
plt.show()

**Create a table showing each K value and its silhouette score.**

In [ ]:
silhouette_table = pd.DataFrame({
    "K": list(silhouette_k_values),
    "Silhouette Score": silhouette_scores
})

silhouette_table.sort_values("Silhouette Score", ascending=False)

**Output Interpretation**

A higher silhouette score usually means better clustering.

However, do not rely only on the highest value. Also consider whether the chosen K makes sense for customer segmentation.

## Create the Final K-Means Model

**Based on the elbow curve and silhouette scores, choose a final K value. Then train a final K-Means model.**

Use `random_state=42` and `n_init=10`.

In [ ]:
final_k = 4

final_kmeans = KMeans(n_clusters=final_k, random_state=RANDOM_STATE, n_init=10)
final_labels = final_kmeans.fit_predict(X_scaled)

print("Final K:", final_k)
print("Final model inertia:", final_kmeans.inertia_)
print("Final silhouette score:", silhouette_score(X_scaled, final_labels))

**Add the final cluster labels to the original dataframe in a new column called `Cluster`.**

In [ ]:
df_clustered = df.copy()
df_clustered["Cluster"] = final_labels

**Check the first five rows after adding the cluster labels.**

In [ ]:
df_clustered.head()

## Cluster Analysis

Now we need to understand what each cluster means.

**Create a summary table using `groupby()` to show the mean values of each feature for each cluster.**

In [ ]:
cluster_summary = df_clustered.groupby("Cluster").mean().round(2)
cluster_summary

**Check how many customers are in each cluster.**

In [ ]:
cluster_counts = df_clustered["Cluster"].value_counts().sort_index()
cluster_counts

## Visualizing the Final Clusters

Since the dataset has many features, we will use PCA to reduce the data into two components only for visualization.

This visualization does not replace the original clustering. It only helps us see the clusters in a 2D plot.

**Use PCA with 2 components and plot the clusters.**

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["Cluster"] = final_labels

plt.figure(figsize=(9, 6))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="Cluster", palette="tab10", alpha=0.75)
plt.title("K-Means Customer Segments Visualized with PCA")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

print("Explained variance by 2 PCA components:", pca.explained_variance_ratio_.sum().round(4))

**Output Interpretation**

The PCA plot gives a simplified 2D view of the clusters.

If the clusters are not perfectly separated, that is normal because the original dataset has many features and the plot only shows two compressed dimensions.

## Final Questions

Answer the following questions:

1. Why is this an unsupervised learning problem?

2. Why did we remove the `CUST_ID` column?

3. Which columns had missing values?

4. How did you handle the missing values?

5. Why is scaling important before applying K-Means?

6. Which K value did you choose? Explain your answer using the elbow method and silhouette score.

7. Based on the cluster summary table, describe each customer segment in your own words.

8. Which cluster may represent high-value customers?

9. Which cluster may represent customers who rely more on cash advance?

10. How can a company use these clusters for marketing strategy?

## Final Questions — Suggested Answers

1. **Why is this an unsupervised learning problem?**  
   This is unsupervised learning because the dataset does not contain a target label showing the real customer segment. The goal is to discover natural groups from customer behavior features.

2. **Why did we remove the `CUST_ID` column?**  
   `CUST_ID` is only an identifier. It does not describe spending, payment, balance, or credit behavior, so using it in K-Means could create meaningless distance calculations.

3. **Which columns had missing values?**  
   Run `df.isnull().sum()` before imputation. In the original Kaggle credit card dataset, the commonly missing columns are usually `MINIMUM_PAYMENTS` and `CREDIT_LIMIT`.

4. **How did you handle the missing values?**  
   Missing values were filled using mean imputation: each missing value was replaced with the mean of its column.

5. **Why is scaling important before applying K-Means?**  
   K-Means uses distance calculations. If the features are not scaled, large-value columns such as `BALANCE`, `PURCHASES`, and `CREDIT_LIMIT` can dominate the clustering and reduce the effect of smaller-scale frequency features.

6. **Which K value did you choose? Explain your answer using the elbow method and silhouette score.**  
   I used **K = 4** as the final value because it usually gives a practical balance between the elbow curve and interpretable customer segments. After running the notebook, confirm that the elbow curve begins to slow around this value and compare it with the silhouette table.

7. **Based on the cluster summary table, describe each customer segment in your own words.**  
   Use the `cluster_summary` table. A good interpretation should compare each cluster by `BALANCE`, `PURCHASES`, `CASH_ADVANCE`, `PAYMENTS`, `CREDIT_LIMIT`, and purchase-frequency columns. For example, one cluster may represent low-activity customers, one may represent purchase-oriented customers, one may represent cash-advance users, and one may represent high-balance/high-limit customers.

8. **Which cluster may represent high-value customers?**  
   The high-value cluster is usually the cluster with the highest `PURCHASES`, `PAYMENTS`, `CREDIT_LIMIT`, and strong purchase frequency. Use `cluster_summary.sort_values("PURCHASES", ascending=False)` to identify it.

9. **Which cluster may represent customers who rely more on cash advance?**  
   The cash-advance cluster is the one with the highest `CASH_ADVANCE` and `CASH_ADVANCE_FREQUENCY`. Use `cluster_summary.sort_values("CASH_ADVANCE", ascending=False)` to identify it.

10. **How can a company use these clusters for marketing strategy?**  
   The company can design different strategies for each segment: rewards and premium offers for high-value purchase customers, financial education or balance-transfer offers for high-balance customers, controlled credit or repayment support for cash-advance users, and activation campaigns for low-activity customers.
